In [6]:
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth import default
import io
import requests
import tempfile

# Autenticação no Google Drive
def authenticate_google_drive():
    print("Autenticando no Google Drive...")
    auth.authenticate_user()
    creds, _ = default()
    drive_service = build('drive', 'v3', credentials=creds)
    return drive_service

# Função para carregar os dados do Google Sheets (CSV)
def load_data_from_google_sheets(url):
    print("Carregando dados da planilha do Google Sheets...")

    # Faz a requisição do CSV
    response = requests.get(url)
    response.raise_for_status()

    # Converte o conteúdo CSV diretamente em um DataFrame do pandas
    df = pd.read_csv(io.StringIO(response.text))

    # Renomeia as colunas para remover espaços indesejados e simplificar o acesso
    df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace(';', '')

    # Seleciona as colunas de interesse
    df = df[['Idade', 'Sexo', 'Impulso_Proprio', 'Impulso_Casal', 'Escrupulo', 'Maquiavelismo']]

    # Substitui vírgulas por pontos decimais
    df = df.replace(',', '.', regex=True)

    # Converte as colunas numéricas para float
    df[['Idade', 'Impulso_Proprio', 'Impulso_Casal', 'Escrupulo', 'Maquiavelismo']] = df[['Idade', 'Impulso_Proprio', 'Impulso_Casal', 'Escrupulo', 'Maquiavelismo']].astype(float)

    # Codifica a variável 'Sexo' como categórica e converte para numérico
    df['Sexo'] = df['Sexo'].replace({'Femenino': 'F', 'Masculino': 'M'}).astype('category')
    df['Sexo'] = df['Sexo'].cat.codes  # Converte 'F' e 'M' para números (0 e 1)

    print("Dados carregados com sucesso!\n")
    return df

# Análise MANCOVA (Multivariate Analysis of Covariance)
def perform_mancova(df):
    print("\nRealizando MANCOVA...")

    # Definindo as variáveis dependentes (Y) e independentes (X)
    Y = df[['Impulso_Proprio', 'Impulso_Casal', 'Escrupulo', 'Maquiavelismo']]
    X = df[['Sexo', 'Idade']]

    # Adiciona uma constante no modelo
    X = sm.add_constant(X)

    # Verifique se as colunas X e Y estão no formato correto (float ou int)
    if not np.issubdtype(X.dtypes.values[0], np.number):
        raise ValueError("A matriz exog contém valores não numéricos.")
    if not np.issubdtype(Y.dtypes.values[0], np.number):
        raise ValueError("A matriz endog contém valores não numéricos.")

    # Realiza a MANCOVA
    mancova_model = MANOVA(endog=Y, exog=X)
    mancova_results = mancova_model.mv_test()

    print(mancova_results)
    return mancova_results

# Testes post-hoc
def perform_post_hoc(df):
    print("\nRealizando testes Post-Hoc (Tukey HSD)...")
    posthoc_results = {}

    # Para cada variável dependente, realizamos o teste post-hoc Tukey HSD
    for variable in ['Impulso_Proprio', 'Impulso_Casal', 'Escrupulo', 'Maquiavelismo']:
        posthoc = pairwise_tukeyhsd(df[variable], df['Sexo'], alpha=0.05)
        print("\nPost-Hoc para {}:".format(variable))
        print(posthoc)
        posthoc_results[variable] = posthoc.summary()

    return posthoc_results

# Salvar arquivo CSV no Google Drive
def save_to_drive(drive_service, filename, content):
    print("\nSalvando arquivo no Google Drive...")

    # Cria um arquivo temporário no disco para salvar o CSV
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv') as temp_file:
        temp_file_name = temp_file.name
        content.to_csv(temp_file, index=False)

    # Define o nome do arquivo e o tipo MIME
    file_metadata = {'name': filename, 'mimeType': 'text/csv'}

    # Cria o MediaFileUpload a partir do arquivo temporário
    media = MediaFileUpload(temp_file_name, mimetype='text/csv')

    # Envia o arquivo para o Google Drive
    file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print(f"Arquivo '{filename}' salvo no Google Drive com ID: {file.get('id')}\n")

# Gera sumário descritivo para salvar
def generate_summary(mancova_results, posthoc_results):
    print("\nGerando sumário descritivo...")

    summary = "MANCOVA Results:\n"
    summary += str(mancova_results)

    summary += "\n\nPost-Hoc Results (Tukey HSD):\n"
    for var, result in posthoc_results.items():
        summary += f"\nPost-Hoc for {var}:\n"
        summary += result.as_text()

    df_summary = pd.DataFrame([summary], columns=["Summary"])
    return df_summary

# Função principal
def main():
    # Autentica no Google Drive
    drive_service = authenticate_google_drive()

    # URL do Google Sheets para o CSV exportado
    url = "https://docs.google.com/spreadsheets/d/1FUx1nRvhRRKhwYDwxUXMutXD-17lQ4I-pSZ27ERkC14/export?format=csv&gid=1161976353"

    # Carrega os dados do Google Sheets
    df = load_data_from_google_sheets(url)

    # Realiza a MANCOVA
    mancova_results = perform_mancova(df)

    # Realiza os testes post-hoc
    posthoc_results = perform_post_hoc(df)

    # Gera e salva o sumário descritivo como CSV no Google Drive
    summary_df = generate_summary(mancova_results, posthoc_results)
    save_to_drive(drive_service, "sumario-Impulso_Proprio-Impulso_Casal-Escrupulo-Maquiavelismo.csv", summary_df)

# Executa a função principal
if __name__ == "__main__":
    main()

Autenticando no Google Drive...
Carregando dados da planilha do Google Sheets...
Dados carregados com sucesso!


Realizando MANCOVA...
                 Multivariate linear model
                                                            
------------------------------------------------------------
           x0           Value  Num DF  Den DF F Value Pr > F
------------------------------------------------------------
          Wilks' lambda 0.2337 4.0000 34.0000 27.8738 0.0000
         Pillai's trace 0.7663 4.0000 34.0000 27.8738 0.0000
 Hotelling-Lawley trace 3.2793 4.0000 34.0000 27.8738 0.0000
    Roy's greatest root 3.2793 4.0000 34.0000 27.8738 0.0000
------------------------------------------------------------
                                                            
------------------------------------------------------------
           x1           Value  Num DF  Den DF F Value Pr > F
------------------------------------------------------------
          Wilks' lambda 0.941